In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime

#Create the Spark Session
spark = SparkSession.builder \
        .appName("Spark with Hive") \
        .enableHiveSupport() \
        .master("local[*]") \
        .getOrCreate()

print("Spark is Working!")



spark_df = spark.read.csv('/Users/solo/Projects/DataEngineering/data/raw/Datasets/online_vs_offline_learning_dataset.csv',header=True,inferSchema=True)

spark_df.printSchema()
spark_df.show(2)
spark_df.count()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/23 22:33:38 WARN Utils: Your hostname, Yashs-Mac-mini.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en1)
26/05/23 22:33:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/23 22:33:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark is Working!
root
 |-- Learning_Mode: string (nullable = true)
 |-- Subject: string (nullable = true)
 |-- Study_Hours: double (nullable = true)
 |-- Retention_Score: integer (nullable = true)
 |-- Focus_Level: integer (nullable = true)
 |-- Exam_Score: integer (nullable = true)

+-------------+-------+-----------+---------------+-----------+----------+
|Learning_Mode|Subject|Study_Hours|Retention_Score|Focus_Level|Exam_Score|
+-------------+-------+-----------+---------------+-----------+----------+
|      Offline|English|        7.7|             51|         96|        70|
|      Offline|English|        6.2|             90|         82|        81|
+-------------+-------+-----------+---------------+-----------+----------+
only showing top 2 rows


1000

In [3]:
print(spark.sparkContext.defaultParallelism)
print(spark_df.rdd.getNumPartitions())

df_new = spark_df.repartition(2)
print(df_new.rdd.getNumPartitions())

10
1
2


In [4]:
from pyspark.sql.functions import *


spark_df.select("Subject", "Study_Hours").show(5)
spark_df.select(col("Subject").alias("Subs")).show(5)

+-------+-----------+
|Subject|Study_Hours|
+-------+-----------+
|English|        7.7|
|English|        6.2|
|English|        1.2|
|   Math|        6.5|
|English|        5.5|
+-------+-----------+
only showing top 5 rows
+-------+
|   Subs|
+-------+
|English|
|English|
|English|
|   Math|
|English|
+-------+
only showing top 5 rows


In [5]:
df_2 = spark_df.withColumn("Attention_Score", col("Study_Hours")*col("Retention_Score"))
df_3 = df_2.withColumnRenamed("Study_Hours","Total_Study_Hours")

df_3.filter(col("subject") == "English").show()

+-------------+-------+-----------------+---------------+-----------+----------+------------------+
|Learning_Mode|Subject|Total_Study_Hours|Retention_Score|Focus_Level|Exam_Score|   Attention_Score|
+-------------+-------+-----------------+---------------+-----------+----------+------------------+
|      Offline|English|              7.7|             51|         96|        70|             392.7|
|      Offline|English|              6.2|             90|         82|        81|             558.0|
|       Online|English|              1.2|             75|         66|        71|              90.0|
|       Online|English|              5.5|             95|         58|        78|             522.5|
|       Online|English|              1.3|             87|         76|        78|113.10000000000001|
|      Offline|English|              1.6|             50|         48|        49|              80.0|
|      Offline|English|              2.6|             46|         84|        60|119.60000000000001|


In [6]:
retention_score = [100,99,98,97,96,95,94,93]
df_3.filter(col("Retention_Score").isin(retention_score).alias("Top Scorers")).show(5)
df_3.filter((col("Retention_Score") > 95) & (col("Exam_Score") > 90)).show(5)
df_4 = df_3.drop("Focus_Level")
df_4.show()

+-------------+-----------+-----------------+---------------+-----------+----------+---------------+
|Learning_Mode|    Subject|Total_Study_Hours|Retention_Score|Focus_Level|Exam_Score|Attention_Score|
+-------------+-----------+-----------------+---------------+-----------+----------+---------------+
|       Online|    English|              5.5|             95|         58|        78|          522.5|
|      Offline|    History|              2.4|             94|         44|        72|          225.6|
|      Offline|Programming|              6.3|             97|         68|        86|          611.1|
|       Online|    History|              7.7|            100|         41|        80|          770.0|
|       Online|       Math|              1.9|            100|         51|        79|          190.0|
+-------------+-----------+-----------------+---------------+-----------+----------+---------------+
only showing top 5 rows
+-------------+-----------+-----------------+---------------+------

In [7]:
df_4.dropDuplicates().show(5)

df_4.orderBy(col('Exam_Score').desc()).show(5)

df_4.orderBy(col('Exam_Score').desc(), col('Total_Study_Hours').asc()).show(5)

+-------------+-------+-----------------+---------------+----------+---------------+
|Learning_Mode|Subject|Total_Study_Hours|Retention_Score|Exam_Score|Attention_Score|
+-------------+-------+-----------------+---------------+----------+---------------+
|      Offline|English|              7.6|             64|        71|          486.4|
|       Online|Science|              7.5|             53|        53|          397.5|
|       Online|Science|              6.0|             87|        82|          522.0|
|      Offline|English|              1.9|             62|        74|          117.8|
|      Offline|   Math|              4.6|             74|        79|          340.4|
+-------------+-------+-----------------+---------------+----------+---------------+
only showing top 5 rows
+-------------+-----------+-----------------+---------------+----------+------------------+
|Learning_Mode|    Subject|Total_Study_Hours|Retention_Score|Exam_Score|   Attention_Score|
+-------------+-----------+

In [8]:
df_4.groupBy('Subject').agg(count('*').alias('Student Count')).orderBy(col('Student Count').desc()).show(5)

df_4.groupBy('Subject').agg(avg('Exam_Score').alias('Subject Average')).orderBy(col('Subject Average').desc()).show(5)

+-----------+-------------+
|    Subject|Student Count|
+-----------+-------------+
|    Science|          215|
|    English|          210|
|    History|          198|
|       Math|          195|
|Programming|          182|
+-----------+-------------+

+-----------+-----------------+
|    Subject|  Subject Average|
+-----------+-----------------+
|    English|74.27142857142857|
|    History|71.57575757575758|
|Programming|71.23076923076923|
|    Science|71.04186046511627|
|       Math| 70.0923076923077|
+-----------+-----------------+



In [11]:
df_4.groupBy('Subject').agg(count('*').alias('Student Count'),avg('Exam_Score').alias('Subject Average')).orderBy(col('Student Count').desc()).show(5)

+-----------+-------------+-----------------+
|    Subject|Student Count|  Subject Average|
+-----------+-------------+-----------------+
|    Science|          215|71.04186046511627|
|    English|          210|74.27142857142857|
|    History|          198|71.57575757575758|
|       Math|          195| 70.0923076923077|
|Programming|          182|71.23076923076923|
+-----------+-------------+-----------------+



In [9]:
accum = spark.sparkContext.accumulator(0)

df_4.foreach(lambda row: accum.add(row['Exam_Score']))

print(accum.value)



71675


26/05/23 23:15:28 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 690668 ms exceeds timeout 120000 ms
26/05/23 23:15:28 WARN SparkContext: Killing executors is not supported by current scheduler.
26/05/23 23:15:29 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$